In [16]:
import pandas as pd
import numpy as np
import import_ipynb
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
import PossionModel
import EloModel
import math
from scipy.stats import poisson as sp_poisson


In [18]:

def norm(s): 
    return str(s).strip().lower().replace(" ", "_")

base = pd.read_excel("League Table (2026).xlsx", sheet_name="Base").copy()
base["home_team"] = base["home_team"].map(norm)
base["away_team"] = base["away_team"].map(norm)

# order games
base["game_num"] = base["game_id"].str.extract(r"(\d+)").astype(int)
base = base.sort_values("game_num").reset_index(drop=True)

teams = sorted(pd.unique(pd.concat([base["home_team"], base["away_team"]])))

def build_elo_features(df, k=20.0, home_adv=50.0, init=1500.0):
    elo = {t: init for t in teams}
    elo_home_pre = []
    elo_away_pre = []
    
    for _, r in df.iterrows():
        h, a = r["home_team"], r["away_team"]
        eh, ea = elo[h], elo[a]
        elo_home_pre.append(eh)
        elo_away_pre.append(ea)

        # result: treat OT win as win (no draws)
        home_win = 1.0 if (r.get("home_win",0)==1 or r.get("home_ot_win",0)==1) else 0.0

        # expected
        exp_h = 1.0 / (1.0 + 10.0 ** (-( (eh + home_adv) - ea ) / 400.0))
        
        # update
        elo[h] = eh + k * (home_win - exp_h)
        elo[a] = ea + k * ((1.0-home_win) - (1.0-exp_h))

    out = df.copy()
    out["elo_home_pre"] = elo_home_pre
    out["elo_away_pre"] = elo_away_pre
    out["elo_diff_pre"] = out["elo_home_pre"] - out["elo_away_pre"]
    return out

base_elo = build_elo_features(base)

base2 = base_elo.copy()

team_rows_home = pd.DataFrame({
    "team": base2["home_team"],
    "opp": base2["away_team"],
    "goals": base2["sum home_goals"].astype(int),
    "is_home": 1,
    "elo_diff_pre": base2["elo_diff_pre"],          # home minus away
})

team_rows_away = pd.DataFrame({
    "team": base2["away_team"],
    "opp": base2["home_team"],
    "goals": base2["sum away_goals"].astype(int),
    "is_home": 0,
    "elo_diff_pre": -base2["elo_diff_pre"],         # away minus home
})

team_df = pd.concat([team_rows_home, team_rows_away], ignore_index=True)
pre = ColumnTransformer([
    ("team", OneHotEncoder(handle_unknown="ignore", drop="first"), ["team"]),  # attack
    ("opp",  OneHotEncoder(handle_unknown="ignore", drop="first"), ["opp"]),   # defense (concede effect)
    ("num", "passthrough", ["is_home", "elo_diff_pre"]),
])

pois_global = Pipeline([
    ("pre", pre),
    ("pois", PoissonRegressor(alpha=0.2, max_iter=20000))
])

X = team_df[["team","opp","is_home","elo_diff_pre"]]
y = team_df["goals"].values
pois_global.fit(X, y)

def predict_match(home_team, away_team, home_elo, away_elo):
    home_team, away_team = norm(home_team), norm(away_team)
    elo_diff = home_elo - away_elo

    lam_h = float(pois_global.predict(pd.DataFrame([{
        "team": home_team,
        "opp": away_team,
        "is_home": 1,
        "elo_diff_pre": elo_diff,
    }]))[0])

    lam_a = float(pois_global.predict(pd.DataFrame([{
        "team": away_team,
        "opp": home_team,
        "is_home": 0,
        "elo_diff_pre": -elo_diff,
    }]))[0])

    probs = PossionModel.predict_lambdas_teamwise(lam_h, lam_a)
    return {"lam_home": lam_h, "lam_away": lam_a, **probs}

def team_power_rating(team, teams, pois_model, elo_by_team=None):
    """
    elo_by_team:
    - None => set elo_diff = 0 for all matchups (structural rating)
    - dict/Series => use current elo values (form rating)
    """
    def win_prob_from_lams(lh, la, kmax=12):
        # truncation at kmax; good enough for ranking
        ks = np.arange(kmax+1)
        fact = np.array([math.factorial(k) for k in ks], dtype=float)
        ph = np.exp(-lh) * (lh**ks) / fact
        pa = np.exp(-la) * (la**ks) / fact
        M = np.outer(ph, pa)
        p_home_reg = np.tril(M, -1).sum()
        p_tie = np.trace(M)
        return float(p_home_reg + 0.5*p_tie)

    ps = []
    for opp in teams:
        if opp == team:
            continue

        elo_t = float(elo_by_team[team]) if elo_by_team is not None else 1500.0
        elo_o = float(elo_by_team[opp])  if elo_by_team is not None else 1500.0
        elo_diff = elo_t - elo_o

        # team home vs opp
        lh = float(pois_model.predict(pd.DataFrame([{
            "team": team, "opp": opp, "is_home": 1,
            "elo_diff_pre": elo_diff
        }]))[0])
        la = float(pois_model.predict(pd.DataFrame([{
            "team": opp, "opp": team, "is_home": 0,
            "elo_diff_pre": -elo_diff
        }]))[0])
        p_home = win_prob_from_lams(lh, la)

        # team away vs opp == 1 - P(opp home vs team)
        lh2 = float(pois_model.predict(pd.DataFrame([{
            "team": opp, "opp": team, "is_home": 1,
            "elo_diff_pre": -elo_diff
        }]))[0])
        la2 = float(pois_model.predict(pd.DataFrame([{
            "team": team, "opp": opp, "is_home": 0,
            "elo_diff_pre": elo_diff
        }]))[0])
        p_opp_home = win_prob_from_lams(lh2, la2)
        p_away = 1.0 - p_opp_home

        ps.append(0.5*(p_home + p_away))

    return float(np.mean(ps))

predict_match("Brazil", "Kazakhstan", EloModel.elo_ratings["brazil"], EloModel.elo_ratings["kazakhstan"])


AttributeError: module 'EloModel' has no attribute 'elo_ratings'